In [44]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_log_error

df = pd.read_csv('../data/features.csv', parse_dates=['date'])
print(f"Veri yüklendi: \n {df.shape} \n {df.columns.tolist()}")


Veri yüklendi: 
 (3000888, 39) 
 ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'transactions', 'tatil_tipi', 'yil', 'ay', 'gun', 'haftanin_gunu', 'hafta_sonu', 'ayin_haftasi', 'lag_7', 'lag_14', 'lag_28', 'rolling_7', 'rolling_14', 'family_enc', 'city_enc', 'state_enc', 'type_enc', 'tatil_enc', 'promosyon_var', 'lag_3', 'lag_21', 'lag_10', 'lag_24', 'rolling_3', 'rolling_10', 'rolling_21', 'rolling_24', 'rolling_28']


In [45]:
#Modele verilecek sütunlar (Feature listesi) (part2-model iyileştirme için yeni lag ve rollingler notebooka eklendi )
features = [
    # Mağaza bilgileri
    'store_nbr', 'cluster',
    'type_enc', 'city_enc', 'state_enc',

    # Ürün bilgileri
    'family_enc',

    # Promosyon
    'promosyon_var', 'onpromotion',

    # Dışsal faktörler
    'dcoilwtico', 'transactions', 'tatil_enc',

    # Tarih değişkenleri
    'yil', 'ay', 'gun', 'haftanin_gunu',
    'hafta_sonu', 'ayin_haftasi',

    # Lag değişkenleri
    'lag_3', 'lag_7', 'lag_10','lag_14','lag_21','lag_24','lag_28',

    # Rolling ortalamalar
    'rolling_3', 'rolling_7','rolling_10','rolling_14','rolling_21','rolling_24','rolling_28'
]
target = 'sales' #tahmin edilecek hedef değişken

print(f"Toplam feature sayısı: {len(features)} \n Kullanılacak olanlar: {features}")


Toplam feature sayısı: 31 
 Kullanılacak olanlar: ['store_nbr', 'cluster', 'type_enc', 'city_enc', 'state_enc', 'family_enc', 'promosyon_var', 'onpromotion', 'dcoilwtico', 'transactions', 'tatil_enc', 'yil', 'ay', 'gun', 'haftanin_gunu', 'hafta_sonu', 'ayin_haftasi', 'lag_3', 'lag_7', 'lag_10', 'lag_14', 'lag_21', 'lag_24', 'lag_28', 'rolling_3', 'rolling_7', 'rolling_10', 'rolling_14', 'rolling_21', 'rolling_24', 'rolling_28']


In [46]:
#Time Series Forecasting -Train / validation split için geçmiş veriler train, test seti validation (kronolojik bölüyoruz)

train_df = df[df['date'] < '2017-08-01'].copy()
valid_df  = df[df['date'] >= '2017-08-01'].copy()

train_df = train_df.dropna(subset=['lag_7','lag_14','lag_28','rolling_7','rolling_14']) # Lag ve rollinglerden gelen NaN satırlarını düşür

print(f"Train boyutu : {train_df.shape}")
print(f"Valid boyutu : {valid_df.shape}")
print(f"Train tarih  : {train_df['date'].min()} - {train_df['date'].max()}")
print(f"Valid tarih  : {valid_df['date'].min()} - {valid_df['date'].max()}")

# X -> feature'lar: model görür öğrenir
# y -> hedef: model bunu tahmin etmeye çalışır
X_train = train_df[features]
y_train = np.log1p(train_df[target])  # RMSLE hesabı için log aldık
X_valid = valid_df[features]
y_valid = np.log1p(valid_df[target])

print(f"X_train: {X_train.shape}")
print(f"X_valid: {X_valid.shape}")

Train boyutu : (2924262, 39)
Valid boyutu : (26730, 39)
Train tarih  : 2013-01-29 00:00:00 - 2017-07-31 00:00:00
Valid tarih  : 2017-08-01 00:00:00 - 2017-08-15 00:00:00
X_train: (2924262, 31)
X_valid: (26730, 31)


In [47]:
# LightGBM - Model kurma/eğitme (LightGBM: zayıf modelleri(karar ağaçlarını birleştirerek komplike model oluşturur))

lgb_train = lgb.Dataset(X_train , label=y_train)
lgb_valid = lgb.Dataset(X_valid , label=y_valid , reference=lgb_train)

parameters = {
    'objective'    : 'regression',   # sayı tahmin ettiğimiz için regression 
    'metric'       : 'rmse',         # hata metriği kaggle rmse kullanıyor 
    'learning_rate': 0.03,           # her adımda ne kadar öğrensin? (küçük olması yavaş ama güvenli)(0.05 genel kabul)(0.01 çok yavaş-0.3 hızlı ama atlayabilir)
    'num_leaves'   : 63,             # ağacın karmaşıklığı (31-genel kabul (büyük -> karmaşık overfitting/küçük->basit underfitting)
    'min_data_in_leaf': 50,          # yaprakta minimum veri sayısı (3M veri satır için 20 okey)
}

#Model eğitimi
model = lgb.train(
    parameters,
    lgb_train,
    num_boost_round  = 4500,         # Kaç tur eğitim yapılsın?
    valid_sets       = [lgb_valid],  # Validation seti takip et (overfitting vs izlemek için)
    callbacks        = [
        lgb.early_stopping(100),      # 50 turda iyileşme yoksa dur (overfittingi önler )
        lgb.log_evaluation(300)      # Her 100 turda bir sonucu yazdır
    ]
)
print(f"En iyi tur: {model.best_iteration}")


Training until validation scores don't improve for 100 rounds
[300]	valid_0's rmse: 0.386899
[600]	valid_0's rmse: 0.380497
[900]	valid_0's rmse: 0.377411
[1200]	valid_0's rmse: 0.375681
[1500]	valid_0's rmse: 0.37439
[1800]	valid_0's rmse: 0.373297
[2100]	valid_0's rmse: 0.372432
[2400]	valid_0's rmse: 0.371824
[2700]	valid_0's rmse: 0.371226
[3000]	valid_0's rmse: 0.370833
[3300]	valid_0's rmse: 0.370394
[3600]	valid_0's rmse: 0.370032
[3900]	valid_0's rmse: 0.369692
[4200]	valid_0's rmse: 0.369364
[4500]	valid_0's rmse: 0.369077
Did not meet early stopping. Best iteration is:
[4498]	valid_0's rmse: 0.369071
En iyi tur: 4498


In [48]:
#RMSLE Hesaplama (0.5 altı iyi/ 0.3 altı mükemmele yakın)

# Validation seti üzerinde tahmin yap
y_pred_log = model.predict(X_valid)

# Log'dan geri çevir — log1p aldık, expm1 ile geri dönüyoruz
y_pred = np.expm1(y_pred_log)
y_gercek = np.expm1(y_valid)

y_pred = np.maximum(y_pred, 0) #negatif tahminler 0 (satış negatif olamaz)

# RMSLE hesapla
rmsle = np.sqrt(mean_squared_log_error(y_gercek, y_pred))
print(f"RMSLE Değeri: {rmsle:.4f}")



RMSLE Değeri: 0.3691


In [ ]:
model.save_model('../submissions/lgb_model.txt') #modeli kaydet(lgb_model.txt olarak)
